In [ ]:
import pandas as pd

In [ ]:
import openpyxl
data = pd.read_excel("marriage_dataset_v2.xlsx")
data.head()

In [ ]:
data = data.drop(columns=['Name', 'Partner_Name', 'Education', 'Partner_Education', 'Annual_Income', 'Partner_Annual_Income'])
data

In [ ]:
from sklearn.preprocessing import LabelEncoder


label_encoder = LabelEncoder()
data['Gender'] = label_encoder.fit_transform(data['Gender'])
data['Partner_Gender'] = label_encoder.fit_transform(data['Partner_Gender'])
data['Intercaste'] = label_encoder.fit_transform(data['Intercaste'])
data['Are_They_Happy'] = label_encoder.fit_transform(data['Are_They_Happy'])
data['Want_To_Marry']= label_encoder.fit_transform(data['Want_To_Marry'])
data['Married']= label_encoder.fit_transform(data['Married'])
data

In [ ]:
from sklearn.preprocessing import OneHotEncoder

onehot_encoder = OneHotEncoder()
encoder = onehot_encoder.fit_transform(data[['Religion', 'Partner_Religion', 'Location_Type', 'Partner_Location_Type']])
onehot_encoder.get_feature_names_out(['Religion', 'Partner_Religion', 'Location_Type', 'Partner_Location_Type'])

In [ ]:
onehot_encoded_df = pd.DataFrame(encoder.toarray(), columns=onehot_encoder.get_feature_names_out(['Religion', 'Partner_Religion', 'Location_Type', 'Partner_Location_Type']))
onehot_encoded_df

In [ ]:
data = pd.concat([data.drop(['Religion', 'Partner_Religion', 'Location_Type', 'Partner_Location_Type'], axis=1), onehot_encoded_df], axis=1)
data

In [ ]:
import pickle
with open('serialized_object/label_encoder.pkl', 'wb') as file:
    pickle.dump(label_encoder, file)
with open('serialized_object/onehot_encoder.pkl', 'wb') as file:
    pickle.dump(onehot_encoder, file)

with open('proccessed_data.pkl', 'wb') as file:
    pickle.dump(data,file)

In [ ]:
data.head(1)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = data.drop('Married', axis=1)
Y = data['Married']

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
scalar = StandardScaler()
X_train = scalar.fit_transform(X_train)
X_test = scalar.transform(X_test)


In [ ]:
with open('serialized_object/scalar.pkl', 'wb') as file:
    pickle.dump(scalar, file)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier



In [ ]:
def create_model(neurons=64, layer=2):
    model = Sequential()
    model.add(Dense(neurons, activation='relu', input_shape=(X_train.shape[1],)))
    for _ in range(layer - 1):
        model.add(Dense(neurons, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))
    
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

In [ ]:
## create a keras classifier
model = KerasClassifier(model=create_model, epochs=10, batch_size=10, verbose=0, neurons=32, layer=2)

In [ ]:
#Define Grid Search parameters
from sklearn.model_selection import GridSearchCV
param_grid = {
    'neurons': [ 32, 64],
    'layer': [2, 3],
    'epochs': [50, 100],
    'batch_size': [10, 20]
}

In [ ]:
#perform grid search
grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=-1, cv=3)
grid_result = grid.fit(X_train, Y_train)

# summarize results
print(f"Best: {grid_result.best_score_} using {grid_result.best_params_}")
##Best: 0.9506252641150539 using {'batch_size': 10, 'epochs': 50, 'layer': 2, 'neurons': 32}